# 🚀 Nexus Audio - Treino com TPU

**Versão otimizada para TPU** = 3-5x mais rápido que GPU!

⚡ Usa `torch_xla` para acelerar o treinamento no Kaggle TPU.

In [ ]:
# ============================================
# 1. SETUP TPU
# ============================================
import os

# Verificar se estamos em TPU
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    USE_TPU = True
    print("✅ TPU detectada!")
except ImportError:
    USE_TPU = False
    print("⚠️ TPU não disponível, usando GPU/CPU")

# Instalar dependências
!pip install -q einops encodec

# Mamba não funciona em TPU, vamos usar fallback
if not USE_TPU:
    !pip install -q causal-conv1d mamba-ssm || echo 'Mamba fallback'

In [ ]:
# ============================================
# 2. IMPORTS E DEVICE
# ============================================
import torch
import torch.nn as nn
import sys
from pathlib import Path
from tqdm.notebook import tqdm
import time

print(f"PyTorch: {torch.__version__}")

# Configurar device
if USE_TPU:
    device = xm.xla_device()
    print(f"🚀 Usando TPU: {device}")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"🎮 Usando GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("💻 Usando CPU")

In [ ]:
# ============================================
# 3. CARREGAR CÓDIGO
# ============================================
sys.path.insert(0, "/kaggle/input/nexus-audio-code")
from src.model import SiMBATherapeutic
print("✅ Código carregado!")

In [ ]:
# ============================================
# 4. CARREGAR TOKENS
# ============================================
# ⚡ TROCAR pelo nome do seu dataset de tokens!
TOKENS_PATH = "/kaggle/input/nexus-fma-tokens"

token_files = list(Path(TOKENS_PATH).glob("**/*.pt"))
print(f"✅ Tokens encontrados: {len(token_files)}")

if token_files:
    sample = torch.load(token_files[0])
    print(f"Shape original: {sample.shape}")
    # Shape deve ser [1, 8, 750] -> [batch, n_codebooks, seq_len]

In [ ]:
# ============================================
# 5. CONFIGURAÇÃO
# ============================================
CONFIG = {
    "model": {
        "d_model": 384,
        "n_layers": 6,
        "d_state": 16,
        "d_conv": 4,
        "expand": 2,
        "max_seq_len": 4096,
    },
    "audio": {
        "sample_rate": 24000,
        "encodec_bandwidth": 6.0,
    },
    "therapeutic": {"use_biofeedback": False},
}

# Hiperparâmetros de treino
BATCH_SIZE = 16 if USE_TPU else 8  # TPU aguenta mais!
LEARNING_RATE = 3e-4
MAX_STEPS = 5000
ACCUM_STEPS = 4
SAVE_EVERY = 500

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# ============================================
# 6. CRIAR MODELO
# ============================================
model = SiMBATherapeutic.from_config(CONFIG)
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parâmetros: {n_params:,}")

In [ ]:
# ============================================
# 7. DATASET (CORRIGIDO!)
# ============================================
from torch.utils.data import Dataset, DataLoader

class TokenDataset(Dataset):
    """Dataset de tokens pré-processados"""
    def __init__(self, token_dir):
        self.files = list(Path(token_dir).glob("**/*.pt"))
        print(f"Dataset: {len(self.files)} tokens")
        
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        tokens = torch.load(self.files[idx])
        # Shape original: [1, 8, 750] -> precisa virar [8, 750]
        if tokens.dim() == 3:
            tokens = tokens.squeeze(0)  # Remove batch dim: [8, 750]
        return tokens.long()  # Retorna [n_codebooks, seq_len]

dataset = TokenDataset(TOKENS_PATH)

# Testar um sample
test_sample = dataset[0]
print(f"Shape após Dataset: {test_sample.shape}")  # Deve ser [8, 750]

# DataLoader otimizado para TPU
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    drop_last=True,  # Importante para TPU!
)

In [ ]:
# ============================================
# 8. FUNÇÃO DE TREINO
# ============================================
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

def train_tpu(model, loader, max_steps, lr, accum=4):
    print(f"\n{'='*50}")
    print(f"🚀 Treinando com {'TPU' if USE_TPU else 'GPU/CPU'}")
    print(f"{'='*50}")
    
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    scheduler = CosineAnnealingLR(optimizer, T_max=max_steps, eta_min=lr*0.1)
    
    # Para TPU, usar ParallelLoader
    if USE_TPU:
        train_loader = pl.ParallelLoader(loader, [device]).per_device_loader(device)
    else:
        train_loader = loader
    
    model.train()
    global_step = 0
    running_loss = 0
    start = time.time()
    
    # Checkpoint inicial
    torch.save(model.state_dict(), f"{CHECKPOINT_DIR}/step0.pt")
    print("💾 Checkpoint inicial salvo!")
    
    pbar = tqdm(total=max_steps, desc="Treinando")
    
    while global_step < max_steps:
        for batch_idx, tokens in enumerate(train_loader):
            # tokens shape: [batch, n_codebooks, seq_len]
            tokens = tokens.to(device)
            
            # Forward
            outputs = model(tokens=tokens, labels=tokens)
            loss = outputs["loss"]
            if isinstance(loss, tuple):
                loss = loss[0]
            loss = loss.mean() / accum
            
            # Backward
            loss.backward()
            running_loss += loss.item()
            
            if (batch_idx + 1) % accum == 0:
                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                
                # TPU precisa de xm.optimizer_step
                if USE_TPU:
                    xm.optimizer_step(optimizer)
                else:
                    optimizer.step()
                
                scheduler.step()
                optimizer.zero_grad()
                
                global_step += 1
                pbar.update(1)
                
                # Log a cada 50 steps
                if global_step % 50 == 0:
                    avg_loss = running_loss / 50
                    pbar.set_postfix({"loss": f"{avg_loss:.4f}"})
                    running_loss = 0
                
                # Salvar checkpoint
                if global_step % SAVE_EVERY == 0:
                    torch.save(model.state_dict(), f"{CHECKPOINT_DIR}/step{global_step}.pt")
                    elapsed = (time.time() - start) / 60
                    print(f"\n💾 Step {global_step} | Tempo: {elapsed:.1f}min")
                
                if global_step >= max_steps:
                    break
        
        # Reiniciar loader se necessário
        if USE_TPU:
            train_loader = pl.ParallelLoader(loader, [device]).per_device_loader(device)
    
    pbar.close()
    elapsed = (time.time() - start) / 60
    print(f"\n✅ Treino completo! Tempo: {elapsed:.1f} min")
    
    # Salvar modelo final
    torch.save(model.state_dict(), f"{CHECKPOINT_DIR}/final.pt")
    return model

In [ ]:
# ============================================
# 9. TREINAR!
# ============================================
model = train_tpu(
    model=model,
    loader=loader,
    max_steps=MAX_STEPS,
    lr=LEARNING_RATE,
    accum=ACCUM_STEPS,
)

In [ ]:
# ============================================
# 10. SALVAR MODELO
# ============================================
final_checkpoint = {
    "model_state_dict": model.state_dict(),
    "config": CONFIG,
}
torch.save(final_checkpoint, f"{CHECKPOINT_DIR}/nexus_trained.pt")

!cp /kaggle/working/checkpoints/*.pt /kaggle/working/
!ls -la /kaggle/working/*.pt
print("\n✅ Modelo salvo!")

---
# 🎵 Testar Geração

In [ ]:
import torchaudio

# Mover modelo para CPU para geração (evita problemas com TPU)
model_cpu = model.to('cpu')
model_cpu.eval()

print("Gerando 10s de música...")
with torch.no_grad():
    waveform = model_cpu.generate(duration_seconds=10.0, temperature=0.9)

if waveform.dim() == 3:
    waveform = waveform.squeeze(0)
torchaudio.save("/kaggle/working/output.wav", waveform.cpu(), 24000)
print("✅ Áudio salvo!")

In [ ]:
from IPython.display import Audio
Audio("/kaggle/working/output.wav")